# 01. RAG 개념 이해

## RAG란?

**RAG(Retrieval-Augmented Generation)** 는 LLM의 한계를 극복하기 위해  
**외부 문서를 검색(Retrieval)** 하고, 그 결과를 바탕으로 **답변을 생성(Generation)** 하는 기술임

---

## LLM의 한계

| 문제 | 설명 |
|---|---|
| 지식 단절 | 학습 데이터 이후의 최신 정보를 모름 |
| 할루시네이션 | 사실과 다른 내용을 그럴듯하게 생성 |
| 출처 불명 | 어디서 얻은 정보인지 알 수 없음 |
| 컨텍스트 제한 | 모든 문서를 프롬프트에 넣을 수 없음 |

---

## RAG 파이프라인

```
【 인덱싱 단계 (오프라인) 】

  문서 로딩  →  텍스트 분할  →  임베딩  →  벡터 스토어 저장

【 검색·생성 단계 (온라인) 】

  사용자 질문  →  임베딩  →  유사 문서 검색  →  LLM 답변 생성
```

---



## Naive RAG 구성 요소

| 단계 | 역할 | 도구 |
|---|---|---|
| Document Loader | 문서 불러오기 | TextLoader, PyPDFLoader |
| Text Splitter | 문서를 청크로 분할 | RecursiveCharacterTextSplitter |
| Embedding | 텍스트 → 벡터 변환 | OpenAIEmbeddings |
| Vector Store | 벡터 저장 및 검색 | FAISS, Chroma |
| Retriever | 관련 문서 검색 | as_retriever() |
| LLM | 최종 답변 생성 | ChatOpenAI |

## RAG vs 일반 LLM 비교 실습

In [1]:
import os
import sys
from dotenv import load_dotenv
sys.path.append("..")  # 상위 폴더의 common_config.py를 import하기 위해 추가
from common_config import get_llm

load_dotenv(override=True, dotenv_path="../.env")

llm = get_llm("gpt-5.4-mini", temperature=0)

In [2]:
# LLM 단독 호출 - 모델이 알고 있는 내용만 답변
question = "LangGraph는 무엇이고 어떤 개념을 사용하나요?"

response = llm.invoke(question)
print("[LLM 단독 답변]")
print(response.content)

[LLM 단독 답변]
LangGraph는 **에이전트(Agent)나 워크플로우를 그래프 구조로 설계하고 실행할 수 있게 해주는 프레임워크**입니다.  
LangChain 생태계의 일부로 많이 사용되며, 특히 **복잡한 LLM 애플리케이션**을 만들 때 유용합니다.

## LangGraph가 하는 일
기존의 단순한 체인(chain) 방식은 보통 순차적으로 한 단계씩 실행됩니다.  
반면 LangGraph는 실행 흐름을 **노드(Node)와 엣지(Edge)** 로 표현해서:

- 조건에 따라 분기하고
- 반복하고
- 여러 에이전트가 협업하고
- 중간 상태를 저장/갱신하고
- 사람이 개입하는 흐름까지

구현할 수 있습니다.

---

## 핵심 개념

### 1. Graph(그래프)
전체 워크플로우를 나타내는 구조입니다.  
노드와 엣지로 구성됩니다.

### 2. Node(노드)
각 단계의 작업 단위입니다. 예를 들면:
- 사용자 질문 분석
- 검색 수행
- 답변 생성
- 결과 검증

각 노드는 보통 하나의 함수나 실행 단위에 대응합니다.

### 3. Edge(엣지)
노드 간 이동 경로입니다.  
“다음에 어떤 노드로 갈지”를 정의합니다.

- **일반 엣지**: A → B
- **조건부 엣지**: 결과에 따라 A → B 또는 A → C

### 4. State(상태)
그래프 전체가 공유하는 데이터입니다.  
예:
- 사용자의 입력
- 검색 결과
- 중간 추론
- 최종 답변

노드들은 이 상태를 읽고 수정합니다.

### 5. Conditional Routing(조건 분기)
실행 중 상태를 보고 다음 노드를 선택합니다.  
예:
- 질문이 단순하면 바로 답변
- 복잡하면 검색 후 답변
- 검증 실패 시 다시 수정

### 6. Cycles / Loops(순환)
LangGraph는 반복 구조를 자연스럽게 지원합니다.  
예:
- “검토 → 수정 → 재검토”
- “도구 사용 → 결과 확인 → 다시 도구 사용”

### 7. Checkpointing(체크포인트)
중간

In [3]:
# 외부 문서를 직접 제공하면 더 정확한 답변 가능
context = """
LangGraph는 LangChain의 확장으로, 그래프 기반의 멀티 에이전트 워크플로우를 구성할 수 있음
State, Node, Edge 개념을 사용해 복잡한 AI 파이프라인을 구현함
"""

prompt = f"다음 문서를 참고해서 질문에 답하세요.\n\n문서: {context}\n\n질문: {question}"

response = llm.invoke(prompt)
print("[문서 기반 답변]")
print(response.content)

[문서 기반 답변]
LangGraph는 **LangChain의 확장 도구**로, **그래프 기반의 멀티 에이전트 워크플로우**를 구성할 수 있게 해줍니다.

사용하는 주요 개념은 다음과 같습니다:
- **State**: 워크플로우의 현재 상태를 저장
- **Node**: 각 작업이나 에이전트를 나타냄
- **Edge**: 노드 간의 흐름과 연결을 정의함

즉, LangGraph는 이 개념들을 사용해 **복잡한 AI 파이프라인**을 구현합니다.


---

## 정리

- RAG는 **외부 문서 → 검색 → 생성** 3단계로 동작함
- 다음 노트북부터 각 단계를 하나씩 실습함

    ```
    02_document_loader   → 문서 불러오기
    03_text_splitter     → 텍스트 분할
    04_embedding         → 임베딩
    05_vector_store      → 벡터 스토어
    06_retriever         → 검색
    07_rag_pipeline      → 전체 파이프라인
    ```